# 🔧 Análise do Dataset Feature-Engineered — Olist Recommender

> Notebook complementar ao `01_eda.ipynb`, focado em validar e analisar as 42 features engenheiradas.

**Projeto**: Olist Recommender System  
**Tech Challenge**: Fase 02 — Pós-graduação ML Engineering  
**Dataset**: `data/processed/interactions_fe.parquet` (99.785 linhas × 42 colunas)

### Pipeline de Origem
```
Raw CSVs → data_preparation.py → interactions.parquet (10 cols)
         ↓
         feature_engineering.py → interactions_fe.parquet (42 cols)  ← ESTE NOTEBOOK
```

### Features por Categoria
- **Identificadores** (5): IDs originais + mapeamentos para embedding
- **Target/Sinal** (3): review_score, has_review, purchase_count
- **Numéricas** (6): price, freight_value, logs, ratios, outliers
- **Temporais** (8): ano, mês, dia, hora, recência, sazonalidade
- **Categóricas Encodadas** (8): target encoding, frequency, OHE payment
- **Agregações Usuário** (6): histórico de compras por usuário
- **Agregações Produto** (6): popularidade e stats por produto

## 📋 Índice

1. [Configuração e Carregamento](#1)
2. [Visão Geral do Dataset Feature-Engineered](#2)
3. [Validação das Transformações Numéricas](#3)
4. [Análise das Features Temporais](#4)
5. [Análise das Features Categóricas Encodadas](#5)
6. [Análise das Agregações de Usuário](#6)
7. [Análise das Agregações de Produto](#7)
8. [Análise de Correlações entre Features](#8)
9. [Validação de Qualidade (nulos, variância)](#9)
10. [Comparação Antes/Depois da FE](#10)
11. [Conclusões e Próximos Passos](#11)

## 1. ⚙️ Configuração e Carregamento<a id="1"></a>

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from loguru import logger
from IPython.display import display, Markdown
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

# Estilo
plt.style.use('ggplot')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.bbox'] = 'tight'

# Paths (relativos ao projeto — execute o notebook a partir de notebooks/)
PROJECT_ROOT = Path('..').resolve()
DATA_PATH = PROJECT_ROOT / 'data' / 'processed' / 'interactions_fe.parquet'
FIGURES_DIR = PROJECT_ROOT / 'reports' / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


## 2. 📐 Visão Geral do Dataset Feature-Engineered<a id="2"></a>

O dataset passou de **10 colunas** (interactions.parquet) para **42 colunas** (interactions_fe.parquet) após o pipeline de Feature Engineering.

In [ ]:
df = pd.read_parquet(DATA_PATH)
display(Markdown(f"**Shape**: {df.shape}"))
display(df.head(3))
display(df.dtypes)

### 2.1 Agrupamento por Categoria

In [ ]:
feature_groups = {
    'Identificadores': ['customer_unique_id', 'product_id', 'user_id', 'product_id_idx', 'category_id'],
    'Target/Sinal': ['review_score', 'has_review', 'purchase_count'],
    'Numéricas': ['price', 'freight_value', 'price_log', 'freight_value_log', 'price_to_freight_ratio', 'has_price_outlier'],
    'Temporais': ['purchase_year', 'purchase_month', 'purchase_day_of_week', 'purchase_hour', 'purchase_day_of_month', 'is_weekend', 'is_holiday_season', 'days_since_reference'],
    'Categóricas Encodadas': ['category_target_enc', 'category_frequency', 'category_is_top10', 'category_is_rare', 'payment_type_credit_card', 'payment_type_boleto', 'payment_type_voucher', 'payment_type_debit_card'],
    'Agregações Usuário': ['user_total_purchases', 'user_avg_price', 'user_avg_freight', 'user_purchase_span_days', 'user_recency_days', 'user_review_rate'],
    'Agregações Produto': ['product_popularity', 'product_unique_buyers', 'product_avg_review_score', 'product_avg_price', 'product_avg_freight', 'product_review_rate']
}

group_counts = {k: len(v) for k, v in feature_groups.items()}
display(pd.DataFrame.from_dict(group_counts, orient='index', columns=['Qtd Features']))

## 3. 📈 Validação das Transformações Numéricas<a id="3"></a>

### 3.1 Log Transformations

Aplicamos `log1p` em `price` e `freight_value` devido à distribuição log-normal identificada no EDA anterior.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

sns.histplot(df['price'], bins=50, ax=axes[0, 0], color='skyblue')
axes[0, 0].set_title('Price Original')

sns.histplot(df['price_log'], bins=50, ax=axes[0, 1], color='steelblue')
axes[0, 1].set_title('Price Log-Transformed')

sns.histplot(df['freight_value'], bins=50, ax=axes[1, 0], color='lightgreen')
axes[1, 0].set_title('Freight Value Original')

sns.histplot(df['freight_value_log'], bins=50, ax=axes[1, 1], color='forestgreen')
axes[1, 1].set_title('Freight Value Log-Transformed')

plt.tight_layout()
fig.savefig(FIGURES_DIR / '09_log_transformations.png')
plt.show()

In [ ]:
display(Markdown("### 3.2 Verificação da Normalização"))
skewness_comparison = pd.DataFrame({
    'Feature Original': ['price', 'freight_value'],
    'Skewness Original': [df['price'].skew(), df['freight_value'].skew()],
    'Feature Transformada': ['price_log', 'freight_value_log'],
    'Skewness Transformada': [df['price_log'].skew(), df['freight_value_log'].skew()]
})
display(skewness_comparison)

### 3.3 Price-to-Freight Ratio

Feature engenheirada: `price / (freight_value + 1)` — mede a eficiência de custo.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
sns.histplot(df['price_to_freight_ratio'], bins=50, ax=ax, color='purple')
ax.set_title('Distribuição de Price-to-Freight Ratio')

plt.tight_layout()
fig.savefig(FIGURES_DIR / '10_price_to_freight_ratio.png')
plt.show()

display(df[['price_to_freight_ratio']].describe().T.round(2))

## 4. 📅 Análise das Features Temporais<a id="4"></a>

Extraímos 8 features temporais do `order_purchase_timestamp`. Vamos validar sua distribuição.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

sns.countplot(data=df, x='purchase_year', ax=axes[0, 0], palette='Set2')
axes[0, 0].set_title('Distribuição por Ano')

sns.countplot(data=df, x='purchase_month', ax=axes[0, 1], palette='coolwarm')
axes[0, 1].set_title('Distribuição por Mês')

sns.countplot(data=df, x='purchase_day_of_week', ax=axes[1, 0], palette='Set2')
axes[1, 0].set_title('Distribuição por Dia da Semana')

sns.histplot(data=df, x='purchase_hour', bins=24, ax=axes[1, 1], color='orange', discrete=True)
axes[1, 1].set_title('Distribuição por Hora do Dia')

plt.tight_layout()
fig.savefig(FIGURES_DIR / '11_temporal_features.png')
plt.show()

### 4.1 Sazonalidade Detectada

Features booleanas: `is_weekend` e `is_holiday_season`.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

df['is_weekend'].value_counts().plot.pie(autopct='%1.1f%%', ax=axes[0], colors=['lightblue', 'salmon'], startangle=90)
axes[0].set_title('Proporção de Compras no Fim de Semana')
axes[0].set_ylabel('')

df['is_holiday_season'].value_counts().plot.pie(autopct='%1.1f%%', ax=axes[1], colors=['lightblue', 'salmon'], startangle=90)
axes[1].set_title('Proporção de Compras em Época de Feriado')
axes[1].set_ylabel('')

plt.tight_layout()
fig.savefig(FIGURES_DIR / '12_seasonality_flags.png')
plt.show()

## 5. 🏷️ Análise das Features Categóricas Encodadas<a id="5"></a>

### 5.1 Target Encoding da Categoria

`category_target_enc` = média de review_score por categoria (com smoothing bayesiano α=10).

In [ ]:
display(df[['category_target_enc']].describe().T.round(3))

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.histplot(df['category_target_enc'].dropna(), bins=30, ax=axes[0], color='teal')
axes[0].set_title('Distribuição do Target Encoding da Categoria')

top10_cat = df.groupby('category_id')['category_target_enc'].mean().sort_values(ascending=False).head(10)
top10_cat.plot.bar(ax=axes[1], color='teal')
axes[1].set_title('Top 10 Categorias por Target Encoding')
axes[1].set_ylabel('Target Encoding')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
fig.savefig(FIGURES_DIR / '13_target_encoding.png')
plt.show()

### 5.2 Frequency Encoding e Flags

- `category_frequency`: frequência normalizada da categoria no dataset
- `category_is_top10`: flag (1 se categoria está no top 10)
- `category_is_rare`: flag (1 se categoria tem < 10 interações)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.histplot(df['category_frequency'], bins=30, ax=axes[0], color='indigo')
axes[0].set_title('Frequência da Categoria')

sns.countplot(data=df, x='category_is_top10', ax=axes[1], palette='Set2')
axes[1].set_title('Flag: Categoria no Top 10')

sns.countplot(data=df, x='category_is_rare', ax=axes[2], palette='Set2')
axes[2].set_title('Flag: Categoria Rara')

plt.tight_layout()
fig.savefig(FIGURES_DIR / '14_category_flags.png')
plt.show()

### 5.3 One-Hot Encoding de payment_type

4 flags binárias: credit_card, boleto, voucher, debit_card.

In [ ]:
payment_flags = ['payment_type_credit_card', 'payment_type_boleto', 'payment_type_voucher', 'payment_type_debit_card']
payment_counts = df[payment_flags].sum()

fig, ax = plt.subplots(figsize=(10, 6))
payment_counts.plot.bar(ax=ax, color='coral')
ax.set_title('Frequência dos Tipos de Pagamento')
ax.set_ylabel('Quantidade de Transações')
ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
fig.savefig(FIGURES_DIR / '15_payment_type_ohe.png')
plt.show()

## 6. 👤 Análise das Agregações de Usuário<a id="6"></a>

6 features calculadas por `customer_unique_id`:
- `user_total_purchases`: total de compras
- `user_avg_price`: preço médio gasto
- `user_avg_freight`: frete médio
- `user_purchase_span_days`: diferença entre 1ª e última compra
- `user_recency_days`: dias desde última compra
- `user_review_rate`: proporção de compras com review

In [ ]:
user_feats = ['user_total_purchases', 'user_avg_price', 'user_avg_freight', 'user_purchase_span_days', 'user_recency_days', 'user_review_rate']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, feat in enumerate(user_feats):
    sns.histplot(df[feat], bins=30, ax=axes[i], color='royalblue')
    axes[i].set_title(f'Distribuição de {feat}')

plt.tight_layout()
fig.savefig(FIGURES_DIR / '16_user_aggregations.png')
plt.show()

In [ ]:
display(df[user_feats].describe().T.round(2))

## 7. 📦 Análise das Agregações de Produto<a id="7"></a>

6 features calculadas por `product_id`:
- `product_popularity`: total de vezes comprado
- `product_unique_buyers`: nº de compradores únicos
- `product_avg_review_score`: média de notas
- `product_avg_price`: preço médio histórico
- `product_avg_freight`: frete médio histórico
- `product_review_rate`: proporção com review

In [ ]:
product_feats = ['product_popularity', 'product_unique_buyers', 'product_avg_review_score', 'product_avg_price', 'product_avg_freight', 'product_review_rate']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, feat in enumerate(product_feats):
    sns.histplot(df[feat], bins=30, ax=axes[i], color='mediumseagreen')
    axes[i].set_title(f'Distribuição de {feat}')

plt.tight_layout()
fig.savefig(FIGURES_DIR / '17_product_aggregations.png')
plt.show()

## 8. 🔗 Análise de Correlações entre Features<a id="8"></a>

Vamos identificar correlações fortes (positivas e negativas) entre as features engenheiradas.

In [ ]:
# Selecionar features numéricas, excluindo IDs
exclude_cols = ['customer_unique_id', 'product_id', 'user_id', 'product_id_idx', 'category_id', 'has_price_outlier']
num_cols = df.select_dtypes(include=[np.number]).columns.drop(exclude_cols, errors='ignore')

corr_matrix = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(20, 16))
sns.heatmap(corr_matrix, cmap='coolwarm', annot=False, fmt='.2f', linewidths=0.5, ax=ax)
ax.set_title('Heatmap de Correlações entre Features')

plt.tight_layout()
fig.savefig(FIGURES_DIR / '18_correlation_heatmap.png')
plt.show()

In [ ]:
# Mostrar top correlações
corr_unstacked = corr_matrix.unstack()
corr_unstacked = corr_unstacked[corr_unstacked < 1.0] # remover diagonal
top_corr = corr_unstacked.abs().sort_values(ascending=False).drop_duplicates().head(10)

display(Markdown("### Top 10 Correlações mais fortes (absolutas)"))
display(pd.DataFrame(top_corr, columns=['Correlação Absoluta']))

## 9. ✅ Validação de Qualidade<a id="9"></a>

Vamos verificar:
- Nulos (apenas `review_score` deve ter)
- Variância zero (não deve haver features constantes)
- Tipos de dados

In [ ]:
# Nulos
nulls = df.isnull().sum()
nulls = nulls[nulls > 0]
display(Markdown("**Colunas com valores nulos:**"))
display(pd.DataFrame(nulls, columns=['Qtd Nulos']))

# Plotar nulos
if len(nulls) > 0:
    fig, ax = plt.subplots(figsize=(8, 4))
    nulls.plot.bar(ax=ax, color='salmon')
    ax.set_title('Quantidade de Valores Nulos por Coluna')
    ax.set_ylabel('Valores Nulos')
    plt.tight_layout()
    fig.savefig(FIGURES_DIR / '19_quality_validation.png')
    plt.show()
else:
    # criar dummy se n tem nulos
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.text(0.5, 0.5, 'Sem Nulos', ha='center', va='center')
    plt.tight_layout()
    fig.savefig(FIGURES_DIR / '19_quality_validation.png')
    plt.show()

# Constant features
constant_features = [col for col in df.columns if df[col].nunique() <= 1]
display(Markdown(f"**Features Constantes:** {constant_features}"))

# Tipos de dados
display(Markdown("**Contagem de Tipos de Dados:**"))
display(df.dtypes.value_counts().to_frame('Qtd'))

## 10. 🔄 Comparação Antes/Depois da Feature Engineering<a id="10"></a>

Comparação direta entre o dataset original (10 colunas) e o engenheirado (42 colunas).

In [ ]:
# Carregar dataset original para comparação
ORIG_DATA_PATH = PROJECT_ROOT / 'data' / 'processed' / 'interactions.parquet'
df_orig = pd.read_parquet(ORIG_DATA_PATH)

# Tabela comparativa
comparison = pd.DataFrame({
    'Dataset Original': [df_orig.shape[0], df_orig.shape[1]],
    'Dataset Feature-Engineered': [df.shape[0], df.shape[1]]
}, index=['Linhas', 'Colunas'])
display(comparison)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.kdeplot(df_orig['price'].dropna(), ax=axes[0], color='gray', label='Original', fill=True, alpha=0.3)
axes[0].set_title('Distribuição de Price (Original)')
axes[0].legend()

sns.kdeplot(df['price_log'].dropna(), ax=axes[1], color='blue', label='Log Transform', fill=True, alpha=0.3)
axes[1].set_title('Distribuição de Price Log')
axes[1].legend()

plt.tight_layout()
fig.savefig(FIGURES_DIR / '20_before_after_comparison.png')
plt.show()

In [ ]:
# Features criadas
orig_cols = set(df_orig.columns)
fe_cols = set(df.columns)
new_cols = fe_cols - orig_cols
kept_cols = fe_cols.intersection(orig_cols)

display(Markdown(f"**Features Mantidas ({len(kept_cols)}):** {', '.join(sorted(kept_cols))}"))
display(Markdown(f"**Novas Features ({len(new_cols)}):** {', '.join(sorted(new_cols))}"))

## 11. 🎯 Conclusões e Próximos Passos<a id="11"></a>

### Validação da Feature Engineering

✅ **42 features geradas** a partir de 10 originais  
✅ **Apenas `review_score` tem nulos** (687 registros, comportamento esperado)  
✅ **Zero features constantes** (após remoção na iteração anterior)  
✅ **Agregações user/product calculadas** para contexto colaborativo

### Features Mais Relevantes (por variância e poder discriminativo)
- `category_target_enc`: captura satisfação média por categoria
- `user_total_purchases`: indicador de engajamento
- `product_popularity`: indicador de demanda
- `days_since_reference`: captura tendência temporal

### Implicações para Modelagem
1. **Modelos de Embedding** (NCF/MLP) se beneficiarão das features user/product_idx
2. **Gradient Boosting** (baseline opcional) pode consumir todas as 42 features
3. **Split temporal** ainda é mandatório — usar `order_purchase_timestamp`
4. **Normalização**: features numéricas têm escalas diferentes — padronizar antes do MLP

### Próximas Etapas
1. Split temporal train/val/test (proporção 70/15/15)
2. Implementação dos baselines (Sklearn)
3. Implementação do MLP/NCF (PyTorch)
4. Pipeline DVC com tracking MLflow
5. Comparação de métricas entre modelos